# F04 HELBRECHT v2
## Camouflage universel + Finalisation YouTube

> *"Helbrecht efface toute trace. Ce qui sort d'ici ne ressemble qu'à une vidéo humaine."*

---

**Ce notebook tourne sur CPU — GPU non requis.**

### Ce que fait cette frégate :
- **Détecte le format** : vertical (Short 9:16) ou horizontal (Long 16:9)
- **Camouflage** : re-encode H.264 CRF18, wipe total métadonnées, loudnorm -14 LUFS YouTube
- **QA automatisée** : audit pré et post-camouflage
- **Chapters YouTube** : générés depuis timing.json
- **Rapport HTML** : livrable opérateur avec toutes les métriques

### Sortie :
- `youtube_short.mp4` (si vertical) ou `youtube_long.mp4` (si horizontal)
- `rapport_f04.html`

### Étapes :
1. Montage Google Drive
2. Vérification FFmpeg
3. Téléchargement des scripts depuis GitHub
4. Configuration des chemins
5. Validation CUSTOS check-out
6. Camouflage + Finalisation
7. Validation CUSTOS check-in
8. Aperçu et téléchargement

---
## Étape 1 — Montage Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('[OK] Google Drive monté sur /content/drive')

---
## Étape 2 — Vérification FFmpeg

> FFmpeg est préinstallé sur Colab. Cette cellule vérifie sa disponibilité.

In [ ]:
import subprocess

result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
if result.returncode == 0:
    print(f'[OK] {result.stdout.splitlines()[0]}')
else:
    print('[INSTALL] ffmpeg absent — installation...')
    subprocess.run(['apt-get', 'install', '-y', '-q', 'ffmpeg'], check=True)
    print('[OK] ffmpeg installé.')

---
## Étape 3 — Téléchargement des scripts depuis GitHub

In [ ]:
import urllib.request, os

REPO_RAW    = 'https://raw.githubusercontent.com/kioka8877-ux/CRUSADER/main'
PROJECT_DIR = '/content/crusader'
os.makedirs(PROJECT_DIR, exist_ok=True)

files_to_download = [
    ('F04_HELBRECHT/CODEBASE/crs_f04_helbrecht.py', 'crs_f04_helbrecht.py'),
    ('CRS_CUSTOS.py',                                'CRS_CUSTOS.py'),
]

for repo_path, local_name in files_to_download:
    url      = f'{REPO_RAW}/{repo_path}'
    dst_path = os.path.join(PROJECT_DIR, local_name)
    urllib.request.urlretrieve(url, dst_path)
    size_kb = os.path.getsize(dst_path) / 1024
    print(f'[OK] {local_name} ({size_kb:.1f} KB)')

print()
print('[OK] Scripts téléchargés.')

---
## Étape 4 — Configuration des chemins

> **Modifiez `DRIVE_BASE` si votre structure Google Drive est différente.**

In [ ]:
import os

# ── MODIFIEZ ICI SI NÉCESSAIRE ────────────────────────────────────────────────
DRIVE_BASE = '/content/drive/MyDrive/DRIVE_CRUSADER'
# ─────────────────────────────────────────────────────────────────────────────

PROJECT_DIR = '/content/crusader'
F04_IN      = os.path.join(DRIVE_BASE, 'F04_HELBRECHT', 'IN')
F04_OUT     = os.path.join(DRIVE_BASE, 'F04_HELBRECHT', 'OUT')
os.makedirs(F04_OUT, exist_ok=True)

print('Configuration :')
print(f'  DRIVE_BASE : {DRIVE_BASE}')
print(f'  F04 IN     : {F04_IN}')
print(f'  F04 OUT    : {F04_OUT}')
print()

# Vérification rapide des fichiers d'entrée
for fname in ['short_render.mp4', 'timing.json']:
    path = os.path.join(F04_IN, fname)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f'  [OK]      {fname} ({size_mb:.1f} MB)')
    else:
        print(f'  [MANQUANT] {fname} — vérifiez F04/IN/')

---
## Étape 5 — Validation CUSTOS check-out (F04)

In [ ]:
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, os.path.join(PROJECT_DIR, 'CRS_CUSTOS.py'),
     '--frigate', 'F04', '--mode', 'check-out', '--drive-base', DRIVE_BASE],
)
if result.returncode != 0:
    print('[STOP] CUSTOS check-out FAIL. Vérifiez short_render.mp4 et timing.json dans F04/IN/.')

---
## Étape 6 — Camouflage + Finalisation YouTube

> **Durée estimée : 2–8 minutes selon la longueur de la vidéo.**
>
> Opérations effectuées :
> - Détection automatique du format (vertical Short / horizontal Long)
> - **Wipe total** des métadonnées source (encoder, software, tool signatures)
> - Re-encode H.264 CRF18 (visually lossless — efface les fingerprints du stream)
> - GOP régulier toutes les 2s (structure standard, aucun artefact outil)
> - Audio AAC 192k 48kHz stéréo + **loudnorm -14 LUFS** (standard YouTube)
> - Injection tags propres : title + date uniquement
> - `+faststart` : MOOV atom en tête pour streaming instantané
> - Timestamp fichier aligné sur la date de production
> - Rapport HTML généré

In [ ]:
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, os.path.join(PROJECT_DIR, 'crs_f04_helbrecht.py'),
     '--input',  F04_IN,
     '--output', F04_OUT],
)
if result.returncode != 0:
    print('[STOP] Camouflage / Finalisation échoué. Consultez les logs ci-dessus.')

---
## Étape 7 — Validation CUSTOS check-in (F04)

In [ ]:
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, os.path.join(PROJECT_DIR, 'CRS_CUSTOS.py'),
     '--frigate', 'F04', '--mode', 'check-in', '--drive-base', DRIVE_BASE],
)
if result.returncode != 0:
    print('[STOP] CUSTOS check-in FAIL — fichier sortie absent ou invalide.')
else:
    print('[OK] Vidéo finale validée — pipeline terminé.')

---
## Étape 8 — Aperçu et téléchargement

In [ ]:
import os, glob
from IPython.display import Video, HTML, display

# Détecte le fichier de sortie (short ou long)
candidates = glob.glob(os.path.join(F04_OUT, 'youtube_*.mp4'))
rapport    = os.path.join(F04_OUT, 'rapport_f04.html')

if not candidates:
    print('[ERREUR] Aucun fichier youtube_*.mp4 trouvé dans OUT/')
else:
    output_path = candidates[0]
    size_mb     = os.path.getsize(output_path) / (1024 * 1024)
    print(f'Fichier  : {os.path.basename(output_path)}')
    print(f'Taille   : {size_mb:.1f} MB')
    if os.path.isfile(rapport):
        print(f'Rapport  : {rapport}')
    print()
    display(Video(output_path, embed=True, width=360))

In [ ]:
# Téléchargement direct depuis Colab
from google.colab import files
if candidates:
    files.download(candidates[0])
if os.path.isfile(rapport):
    files.download(rapport)